# Replay a Game Session

This notebook demonstrates three workflows:

1. **Export from database** — load a completed game from the SQLite database into a `RecordedGame`.
2. **Hand-written fixture** — create a `RecordedGame` directly for regression testing specific scenarios.
3. **Replay against live TMDb** — run each move through `validate_move` and compare actual outcomes against expected.

## Prerequisites

- `secrets/.env` with `TMDB_API_KEY` (required for replay).
- At least one completed game in the database, or a hand-written fixture.

In [ ]:
from cinema_game_backend.env import load_cinema_game_env

load_cinema_game_env()

## 1. Export a game from the database

Skip this section if using a hand-written fixture.

In [ ]:
from cinema_game_backend.experiments.export import list_game_ids, export_game

for g in list_game_ids(limit=10):
    print(
        f"{g['game_id']}  {g['difficulty']:8s}  {g['status']:12s}  "
        f"{g['start_actor']} -> {g['end_actor']}  ({g['moves']} moves)"
    )

In [ ]:
GAME_ID = ""  # <-- paste a game_id from above
assert GAME_ID, "Set GAME_ID before running this cell."

game = export_game(GAME_ID)

print(f"Start: {game.start_actor}")
print(f"End:   {game.end_actor}")
print(f"Moves: {len(game.moves)}")
for i, m in enumerate(game.moves, 1):
    print(f"  {i}. [{m.movie}] {m.actor}  (expected valid={m.expected.valid})")

## 2. Hand-written game fixture

You can create a `RecordedGame` directly without the database.
This is useful for regression tests against specific scenarios.

In [ ]:
from cinema_game_backend.models.experiment import (
    RecordedGame,
    RecordedMove,
    ExpectedSuccess,
    ExpectedFailure,
)

game = RecordedGame(
    start_actor="Brad Pitt",
    end_actor="Colin Firth",
    moves=[
        RecordedMove(
            movie="12 Years a Slave",
            actor="Michael Fassbender",
            expected=ExpectedSuccess(
                movie_id=76203,
                movie_title="12 Years a Slave",
                actor_id=17288,
                actor_name="Michael Fassbender",
            ),
        ),
        RecordedMove(
            movie="Batman",
            actor="Jack Nicholson",
            expected=ExpectedFailure(),
        ),
    ],
)

game

## 3. Replay against live TMDb

Run each move through `validate_move` using the real TMDb API
and compare outcomes against expectations.

In [ ]:
from cinema_game_backend.config import create_tmdb_client
from cinema_game_backend.experiments.replay import replay_game

tmdb = create_tmdb_client()
results = await replay_game(tmdb, game)

for r in results:
    status = "PASS" if r.passed else "FAIL"
    print(f"  [{status}] [{r.move.movie}] {r.move.actor}: {r.detail}")

passed = sum(1 for r in results if r.passed)
print(f"\n{passed}/{len(results)} moves match expected outcomes.")